# Fidelidad de textura en el régimen NO saturado (Hex bajo)

**Física del sistema (no es un defecto del modelo):**
- Para **Hex alto la saturación es real**: ⟨sz⟩ → uniforme, **no hay textura** que reproducir.
  La saturación aparece aprox. en **Hex ≈ 0.8–0.9**; por debajo de ese umbral el sistema NO
  está saturado y es donde aparecen las **texturas** (skyrmiones, dominios, desorden de espín).
- El R² global mejoró, pero está **dominado por la mayoría saturada** (fácil). No dice si el
  DDPM reproduce bien las **texturas del régimen de campo bajo**, que es lo que interesa.

**Objetivo de este notebook:** *evaluar* (no modificar el sampler) qué tan fielmente el modelo
actual reproduce las texturas en el **régimen no saturado `Hex ≤ HEX_SAT = 0.80`**:
1. Confirmar el inicio de la saturación con los datos (M y σ_M vs Hex).
2. Comparar observables de textura generado-vs-referencia **restringido a Hex ≤ 0.80**
   (P_HF, σ_M, ξ, M) con distancia de Wasserstein y curvas por bin de Hex.
3. R² separado por régimen (texturado vs saturado).
4. Paneles visuales (cmap **jet**) de muestras texturadas: referencia vs generada.

> Generación con **condicionamiento clamp** (identidad dentro de rango ⇒ condicionamiento fiel
> a campo bajo). Sin dynamic thresholding ni muestreo ancestral: aquí se *evalúa* el modelo tal cual.

> **Kaggle:** agrega los 4 datasets desde **+ Add Data** antes de ejecutar.


In [ ]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
pip("scikit-image")
pip("scipy")

import os, math, time, gc, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import wasserstein_distance
from skimage.metrics import structural_similarity as ssim_fn

for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device : {DEVICE}")
print(f"TensorFlow     : {tf.__version__}")
print(f"PyTorch        : {torch.__version__}")


In [ ]:
# ── Rutas Kaggle ──────────────────────────────────────────────────────────────
BASE = "/kaggle/input/datasets/carloscanamejoy"
DATASET_ORIG_PATH = Path(f"{BASE}/dataset-spines-united-v2/dataset_unificado_v2.npz")
DATASET_EXT_PATH  = Path(f"{BASE}/dataset-external-hez-new/dataset-hez-external.npz")
XCEPTION_WEIGHTS  = Path(f"{BASE}/weights-xception-model/modelo_xception_fulldatabaseV3100.h5")
DDPM_CHECKPOINT   = Path(f"{BASE}/weights-models/ddpm_spines_final_39crop.pt")

for p in [DATASET_ORIG_PATH, DATASET_EXT_PATH, XCEPTION_WEIGHTS, DDPM_CHECKPOINT]:
    print(f"[{'OK' if p.exists() else 'FALTA'}]  {p}")

# ── Constantes ────────────────────────────────────────────────────────────────
PARAM_NAMES   = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]
ACTIVE_PARAMS = ["T", "Kan1", "KanS", "Hex", "KDM"]
ACTIVE_IDX    = [PARAM_NAMES.index(p) for p in ACTIVE_PARAMS]
HEX_IDX       = PARAM_NAMES.index("Hex")   # 6
GRID          = 39
IMG_SIZE      = 40
RD_PIXELS     = 18.3
SEED          = 42

N_EVAL        = 500
DDPM_STEPS    = 100
BATCH_GEN     = 64
BATCH_XCEP    = 32

# Umbral físico de saturación: por debajo => régimen texturado (no saturado)
HEX_SAT       = 0.80

rng_eval = np.random.RandomState(7)


In [ ]:
# ── Cargar datasets ───────────────────────────────────────────────────────────
data_orig   = np.load(DATASET_ORIG_PATH, mmap_mode="r")
imgs_orig   = data_orig["img"]
params_orig = np.asarray(data_orig["params"])
N_orig      = imgs_orig.shape[0]

data_ext    = np.load(DATASET_EXT_PATH, mmap_mode="r")
imgs_ext    = data_ext["img"]
params_ext  = np.asarray(data_ext["params"])
N_ext       = imgs_ext.shape[0]

print(f"Original : {N_orig:,} muestras  imgs={imgs_orig.shape}")
print(f"Externo  : {N_ext:,}  muestras  imgs={imgs_ext.shape}")

# ── Scalers sobre SOLO el train del dataset ORIGINAL ─────────────────────────
all_idx = np.arange(N_orig)
idx_tr_pool, idx_te, _, _ = train_test_split(all_idx, params_orig, test_size=0.15, random_state=SEED)
idx_tr, idx_val, _, _     = train_test_split(idx_tr_pool, params_orig[idx_tr_pool], test_size=0.1765, random_state=SEED)
scaler_inv  = MinMaxScaler().fit(params_orig[idx_tr])

idx_all_ddpm = np.arange(N_orig)
idx_tr_ddpm, _ = train_test_split(idx_all_ddpm, test_size=0.30, random_state=SEED)
scaler_ddpm = MinMaxScaler().fit(params_orig[idx_tr_ddpm])

sample40 = np.pad(
    np.asarray(imgs_orig[idx_tr_ddpm[:5000]]).astype(np.float32)[..., 0],
    ((0,0),(0,1),(0,1)), mode='reflect'
)
MN, MX = float(sample40.min()), float(sample40.max())

HEX_MAX_TRAIN = scaler_ddpm.data_max_[HEX_IDX]
print(f"\nscaler_ddpm Hex → [{scaler_ddpm.data_min_[HEX_IDX]:.4f}, {scaler_ddpm.data_max_[HEX_IDX]:.4f}]")
print(f"Hex externo      → [{params_ext[:, HEX_IDX].min():.4f}, {params_ext[:, HEX_IDX].max():.4f}]")
print(f"Umbral saturación HEX_SAT = {HEX_SAT}")
print(f"MN={MN:.4f}  MX={MX:.4f}")

# Subconjunto de evaluación
eval_idx = rng_eval.choice(N_ext, N_EVAL, replace=False)
y_eval   = params_ext[eval_idx].astype(np.float32)
imgs_ref = np.asarray(imgs_ext[eval_idx]).astype(np.float32)
imgs_ref_39   = imgs_ref[..., 0]
imgs_ref_norm = (imgs_ref_39 - MN) / (MX - MN + 1e-8) * 2 - 1

n_tex = int(np.sum(y_eval[:, HEX_IDX] <= HEX_SAT))
print(f"Eval: {N_EVAL} muestras · texturadas (Hex<={HEX_SAT}): {n_tex} · saturadas: {N_EVAL-n_tex}")


In [ ]:
# ── Arquitectura DDPM (idéntica al checkpoint) ────────────────────────────────
def sinusoidal_embedding(t, dim):
    half  = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half-1))
    args  = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
    def forward(self, t, cond):
        return self.t_mlp(sinusoidal_embedding(t, self.t_mlp[0].in_features)) + self.c_mlp(cond)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch);   self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch);  self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout  = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip     = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, emb):
        h = F.silu(self.norm1(x)); h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h)); h = self.dropout(h); h = self.conv2(h)
        return h + self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv  = nn.Conv2d(ch, ch*3, 1); self.proj = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        B, C, H, W = x.shape; h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        q = q.reshape(B,C,-1); k = k.reshape(B,C,-1); v = v.reshape(B,C,-1)
        attn = torch.softmax(torch.bmm(q.transpose(1,2), k) / math.sqrt(C), dim=-1)
        return x + self.proj(torch.bmm(v, attn.transpose(1,2)).reshape(B, C, H, W))

class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1,2,4),
                 cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs = [base_ch*m for m in ch_mults]
        self.emb     = TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)
        self.down_blocks  = nn.ModuleList(); self.down_samples = nn.ModuleList(); self.skip_channels = []
        in_ch = chs[0]
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                                                   ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn   = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.up_blocks  = nn.ModuleList(); self.up_samples = nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i+1)]
            self.up_blocks.append(nn.ModuleList([ResBlock(in_ch+skip_ch, out_ch, emb_dim, dropout=dropout),
                                                 ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.norm_out = nn.GroupNorm(8, chs[0])
        self.conv_out = nn.Conv2d(chs[0], img_channels, 1)
    def forward(self, x, t, cond):
        emb = self.emb(t, cond); h = self.conv_in(x); skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb); skips.append(h); h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, sk in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, sk], dim=1); h = rb1(h, emb); h = rb2(h, emb); h = us(h)
        return self.conv_out(F.silu(self.norm_out(h)))

class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule="cosine", device="cpu"):
        self.T = T
        if schedule == "cosine":
            steps = T+1; s = 0.008
            x  = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x/T)+s)/(1+s)*math.pi*0.5)**2; ac = ac/ac[0]
            betas = (1-ac[1:]/ac[:-1]).clamp(max=0.999)
        else:
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas; ac = torch.cumprod(alphas, dim=0)
        self.betas = betas; self.alphas_cumprod = ac
        self.sqrt_ac   = ac.sqrt()
        self.sqrt_1mac = (1.0 - ac).sqrt()

@torch.no_grad()
def ddpm_sample(model, cond_t, scheduler, n_steps=100):
    """Sampler del modelo tal cual (determinista + clamp)."""
    dev = next(model.parameters()).device
    cond_t = cond_t.to(dev)
    B = cond_t.shape[0]
    x = torch.randn(B, 1, IMG_SIZE, IMG_SIZE, device=dev)
    sqrt_ac   = scheduler.sqrt_ac.to(dev)
    sqrt_1mac = scheduler.sqrt_1mac.to(dev)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_t  = torch.full((B,), t_val, device=dev, dtype=torch.long)
        eps  = model(x, t_t, cond_t)
        sa   = sqrt_ac[t_val]; s1a = sqrt_1mac[t_val]
        x0   = ((x - s1a*eps) / sa).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            x = sqrt_ac[t_prev]*x0 + sqrt_1mac[t_prev]*eps
        else:
            x = x0
    return x[:, 0].cpu().numpy()

print("Arquitectura DDPM definida.")


In [ ]:
# ── Cargar DDPM ───────────────────────────────────────────────────────────────
ckpt = torch.load(DDPM_CHECKPOINT, map_location=DEVICE, weights_only=False)
hp   = ckpt["hyperparams"]
print(f"Hiperparámetros DDPM: {hp}")

ddpm = ConditionalUNet(
    img_channels=1, base_ch=hp["base_ch"], ch_mults=(1,2,4),
    cond_dim=8, emb_dim=hp["cond_emb_dim"], dropout=0.0,
).to(DEVICE)
state = ckpt["ema"] if ("ema" in ckpt and ckpt["ema"] is not None) else ckpt["model"]
ddpm.load_state_dict(state); ddpm.eval()

MODEL_DEVICE = next(ddpm.parameters()).device
print(f"DDPM loaded on : {MODEL_DEVICE}")
scheduler = DDPMScheduler(T=1000, schedule=hp["beta_schedule"], device=MODEL_DEVICE)
print(f"DDPM cargado (EMA={'ema' in ckpt and ckpt['ema'] is not None}).")

# ── Cargar Xception ───────────────────────────────────────────────────────────
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, BatchNormalization, Dropout
from tensorflow.keras.models import Model

def build_xception(n_out=8):
    inp  = Input(shape=(224, 224, 3))
    base = Xception(weights=None, include_top=False, input_tensor=inp)
    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization(name="batch_normalization_4")(x)
    x = Dropout(0.4, name="dropout")(x)
    x = Dense(256, activation="relu", name="dense")(x)
    x = BatchNormalization(name="batch_normalization_5")(x)
    x = Dropout(0.3, name="dropout_1")(x)
    return Model(inp, Dense(n_out, activation="linear", name="dense_1")(x))

with tf.device("/cpu:0"):
    xception = build_xception()
    xception.load_weights(XCEPTION_WEIGHTS)
print(f"Xception cargado. Params={xception.count_params():,}")


In [ ]:
# ── Utilidades: generación, inferencia y visualización (cmap jet) ─────────────
IMG_CMAP = "jet"

def cond_clamp(params):
    """Clamp al rango del scaler (identidad dentro de rango => fiel a campo bajo)."""
    p = params.copy().astype(np.float32)
    p = np.clip(p, scaler_ddpm.data_min_, scaler_ddpm.data_max_)
    return scaler_ddpm.transform(p)

def generate_batch(cond_norm, n_steps=DDPM_STEPS):
    imgs_out = []
    for s in range(0, len(cond_norm), BATCH_GEN):
        c = torch.tensor(cond_norm[s:s+BATCH_GEN], dtype=torch.float32)
        imgs_out.append(ddpm_sample(ddpm, c, scheduler, n_steps=n_steps))
    return np.concatenate(imgs_out, axis=0)

def xception_infer(imgs_40, scaler_out=None):
    if scaler_out is None: scaler_out = scaler_inv
    imgs = imgs_40[:, :39, :39].astype(np.float32)
    imgs = (imgs + 1.0) / 2.0 * (MX - MN) + MN
    results = []
    with tf.device("/cpu:0"):
        for s in range(0, len(imgs), BATCH_XCEP):
            chunk = imgs[s:s+BATCH_XCEP][..., None]
            chunk = tf.image.resize(chunk, (224, 224))
            chunk = tf.image.grayscale_to_rgb(chunk)
            y_n   = xception.predict(chunk, verbose=0)
            results.append(scaler_out.inverse_transform(y_n))
    return np.concatenate(results, axis=0)

def r2_table(y_true, y_pred, label=""):
    rows = []
    for j, name in zip(ACTIVE_IDX, ACTIVE_PARAMS):
        yt = y_true[:, j]; yp = y_pred[:, j]
        if yt.std() < 1e-8: continue
        rows.append({"param": name, "R2": r2_score(yt, yp), "MAE": mean_absolute_error(yt, yp)})
    df = pd.DataFrame(rows)
    print(f"\n{'='*55}\n  {label}\n{'='*55}")
    print(f"  {'param':<6} {'R2':>8} {'MAE':>10}")
    for _, row in df.iterrows():
        flag = " OK" if row.R2 > 0.7 else (" ~" if row.R2 > 0.3 else " X")
        print(f"  {row.param:<6} {row.R2:8.4f} {row.MAE:10.4f}{flag}")
    print(f"  {'mean':>6} {df.R2.mean():8.4f} {df.MAE.mean():10.4f}")
    return df

def show_samples(imgs_gen, imgs_ref_norm, y_params, label,
                 n_tex=6, n_sat=2, fname=None):
    """Panel Referencia | Generada | |Diff| (cmap jet). Prioriza muestras texturadas (Hex<=HEX_SAT)."""
    tex_idx = np.where(y_params[:, HEX_IDX] <= HEX_SAT)[0]
    sat_idx = np.where(y_params[:, HEX_IDX] >  HEX_SAT)[0]
    def pick(idx, n):
        if len(idx) == 0: return np.array([], dtype=int)
        step = max(1, len(idx) // n)
        return idx[::step][:n]
    sel_tex = pick(tex_idx, n_tex); sel = np.concatenate([sel_tex, pick(sat_idx, n_sat)])
    n_total = len(sel)
    if n_total == 0:
        print(f"[show_samples] sin muestras para '{label}'"); return
    fig, axes = plt.subplots(n_total, 3, figsize=(3*2.8, n_total*2.6), squeeze=False)
    fig.suptitle(label, fontsize=13, fontweight="bold", y=1.01)
    for ax, ct in zip(axes[0], ["Reference (external)", "Generated (DDPM)", "|Difference|"]):
        ax.set_title(ct, fontsize=10, fontweight="bold")
    kw_img  = dict(cmap=IMG_CMAP, vmin=-1, vmax=1, interpolation="nearest")
    kw_diff = dict(cmap="hot",    vmin=0,  vmax=1, interpolation="nearest")
    for row, idx in enumerate(sel):
        ref = imgs_ref_norm[idx]; gen = imgs_gen[idx, :39, :39]; diff = np.abs(ref - gen)
        hx = y_params[idx, HEX_IDX]; regime = "SAT" if hx > HEX_SAT else "tex"
        row_label = (f"Hex={hx:.3f} [{regime}]  T={y_params[idx, PARAM_NAMES.index('T')]:.1f}  "
                     f"KDM={y_params[idx, PARAM_NAMES.index('KDM')]:.3f}")
        im0 = axes[row,0].imshow(ref, **kw_img); axes[row,1].imshow(gen, **kw_img)
        im2 = axes[row,2].imshow(diff, **kw_diff)
        axes[row,2].set_xlabel(f"MAE={float(diff.mean()):.3f}  SSIM={ssim_fn(ref,gen,data_range=2.0):.3f}", fontsize=8)
        axes[row,0].set_ylabel(row_label, fontsize=7, rotation=0, labelpad=130, va="center")
        for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            plt.colorbar(im0, ax=axes[row,1], fraction=0.046, pad=0.04)
            plt.colorbar(im2, ax=axes[row,2], fraction=0.046, pad=0.04)
    if 0 < len(sel_tex) < n_total:
        div_y = (len(sel_tex) - 0.5) / n_total
        fig.add_artist(plt.Line2D([0.08,0.92],[1-div_y,1-div_y], transform=fig.transFigure,
                                  color="gray", linewidth=1.2, linestyle="--"))
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=130, bbox_inches="tight"); print(f"Guardado: {fname}")
    plt.show()

print("Utilidades definidas (cmap = jet).")


---
## Generación (modelo tal cual)
Condicionamiento **clamp** (identidad dentro de rango). El régimen texturado `Hex ≤ 0.80`
está muy por debajo de `HEX_MAX_TRAIN`, así que el condicionamiento ahí es fiel.


In [ ]:
print("Generando imágenes (clamp, modelo tal cual)...")
t0 = time.time()
cond = cond_clamp(y_eval)
imgs_gen = generate_batch(cond)
print(f"  OK en {time.time()-t0:.1f}s")

# R2 por régimen — muestra el número del régimen texturado por separado
y_pred = xception_infer(imgs_gen)
mask_tex = y_eval[:, HEX_IDX] <= HEX_SAT
mask_sat = ~mask_tex
print(f"\nTexturadas (Hex<={HEX_SAT}): {mask_tex.sum()} · Saturadas: {mask_sat.sum()}")
df_all = r2_table(y_eval,            y_pred,            "R2 — TODAS las muestras")
df_tex = r2_table(y_eval[mask_tex],  y_pred[mask_tex],  f"R2 — TEXTURADO (Hex<={HEX_SAT})")
df_sat = r2_table(y_eval[mask_sat],  y_pred[mask_sat],  f"R2 — SATURADO  (Hex>{HEX_SAT})")


---
## Paso 1 — Confirmar el inicio de la saturación desde los datos
Curvas de **M** (magnetización media) y **σ_M** (desorden) vs Hex para las imágenes de
**referencia**. La saturación se ve como M→máximo y σ_M→0. Se marca `HEX_SAT = 0.80`.


In [ ]:
# ── Observables físicos (textura) ─────────────────────────────────────────────
cy = cx = GRID // 2
Y, X = np.ogrid[:GRID, :GRID]
DISK = (X-cx)**2 + (Y-cy)**2 <= RD_PIXELS**2

def physical_observables(img39):
    s = img39.astype(np.float64); s_disk = s[DISK]
    M = float(s_disk.mean()); sigma = float(s_disk.std())
    DYf = np.arange(GRID); DYf = np.where(DYf > GRID//2, DYf-GRID, DYf)
    DXf = np.arange(GRID); DXf = np.where(DXf > GRID//2, DXf-GRID, DXf)
    DYm, DXm = np.meshgrid(DYf, DXf, indexing="ij"); r_map = np.sqrt(DYm**2 + DXm**2)
    # correlación espacial a r~5px (orden de corto alcance)
    s_c = (s - M) * DISK
    G   = np.real(np.fft.ifft2(np.abs(np.fft.fft2(s_c))**2)) / max(DISK.sum(), 1)
    mask_r = (r_map >= 4) & (r_map <= 6)
    xi = float(G[mask_r].mean()) if mask_r.any() else 0.0
    # potencia en altas frecuencias
    F2 = np.fft.fftshift(np.fft.fft2(s)); mag = np.abs(F2)**2
    P_HF = float(mag[r_map > GRID/4].sum() / (mag.sum() + 1e-12))
    return {"M": M, "sigma": sigma, "xi": xi, "P_HF": P_HF}

def batch_obs(imgs, crop=True):
    out = []
    for img in imgs:
        if img.ndim == 3: img = img[..., 0]
        if crop and img.shape[0] == 40: img = img[:39, :39]
        out.append(physical_observables(img))
    return pd.DataFrame(out)

obs_ref = batch_obs(imgs_ref_norm, crop=False)
obs_gen = batch_obs(imgs_gen, crop=True)
hexv = y_eval[:, HEX_IDX]

nbins = 12
edges = np.quantile(hexv, np.linspace(0, 1, nbins+1)); edges[-1] += 1e-6
centers = 0.5*(edges[:-1]+edges[1:])
def binned(series):
    m = np.full(nbins, np.nan); sd = np.full(nbins, np.nan)
    for b in range(nbins):
        sel = (hexv >= edges[b]) & (hexv < edges[b+1])
        if sel.any(): m[b] = np.nanmean(series[sel]); sd[b] = np.nanstd(series[sel])
    return m, sd

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, obs, ttl in zip(axes, ["M", "sigma"], ["M  (magnetización media)", "σ_M  (desorden de espín)"]):
    m, sd = binned(obs_ref[obs].values)
    ax.plot(centers, m, "-o", color="steelblue", lw=2, ms=4, label="Referencia ext.")
    ax.fill_between(centers, m-sd, m+sd, color="steelblue", alpha=0.15)
    ax.axvline(HEX_SAT, color="crimson", ls="--", lw=1.5, label=f"HEX_SAT={HEX_SAT}")
    ax.set_xlabel("Hex"); ax.set_ylabel(obs); ax.set_title(ttl); ax.legend(fontsize=8)
fig.suptitle("Paso 1: inicio de la saturación en la REFERENCIA (M↑, σ_M↓)",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig("step1_saturation_onset.png", dpi=140); plt.show()


---
## Paso 2 — Fidelidad de textura en el régimen NO saturado (Hex ≤ 0.80)
Solo muestras texturadas. Distribuciones generado-vs-referencia de los observables de
textura (σ_M, ξ, P_HF) y distancia de Wasserstein. Curvas por bin de Hex dentro del régimen.


In [ ]:
# ── Restringir al régimen texturado ───────────────────────────────────────────
tex = (hexv <= HEX_SAT)
obs_names = ["sigma", "xi", "P_HF"]
obs_labels = {"sigma": "σ_M (desorden)", "xi": "ξ (correlación r≈5px)", "P_HF": "P_HF (alta frec.)"}

print(f"Wasserstein generado-vs-referencia · SOLO texturado (Hex<={HEX_SAT}, n={int(tex.sum())})")
print(f"  {'obs':<8} {'W(tex)':>12}    [referencia para contexto]")
for o in obs_names:
    w = wasserstein_distance(obs_ref[o][tex], obs_gen[o][tex])
    print(f"  {o:<8} {w:12.6f}")

# Histogramas (solo texturado)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, o in zip(axes, obs_names):
    lo = min(obs_ref[o][tex].min(), obs_gen[o][tex].min())
    hi = max(obs_ref[o][tex].max(), obs_gen[o][tex].max())
    bins = np.linspace(lo, hi, 40)
    ax.hist(obs_ref[o][tex], bins=bins, alpha=0.55, color="steelblue", density=True, label="Referencia")
    ax.hist(obs_gen[o][tex], bins=bins, alpha=0.45, color="green",     density=True, label="Generada")
    ax.set_title(obs_labels[o]); ax.legend(fontsize=8)
fig.suptitle(f"Paso 2: distribuciones de textura · régimen NO saturado (Hex<={HEX_SAT})",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig("step2_texture_hist_nonsat.png", dpi=140); plt.show()

# Curvas por bin de Hex (ref vs gen) dentro del régimen texturado
tex_edges = np.linspace(hexv.min(), HEX_SAT, 7); tex_centers = 0.5*(tex_edges[:-1]+tex_edges[1:])
def binned_tex(series):
    m = np.full(len(tex_centers), np.nan)
    for b in range(len(tex_centers)):
        sel = (hexv >= tex_edges[b]) & (hexv < tex_edges[b+1])
        if sel.any(): m[b] = np.nanmean(series[sel])
    return m

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, o in zip(axes, obs_names):
    ax.plot(tex_centers, binned_tex(obs_ref[o].values), "-o", color="steelblue", lw=2, label="Referencia")
    ax.plot(tex_centers, binned_tex(obs_gen[o].values), "-s", color="green",     lw=2, label="Generada")
    ax.set_xlabel("Hex"); ax.set_title(obs_labels[o]); ax.legend(fontsize=8)
fig.suptitle(f"Paso 2: textura vs Hex dentro del régimen NO saturado (Hex<={HEX_SAT})",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig("step2_texture_vs_hex_nonsat.png", dpi=140); plt.show()


---
## Paso 3 — Inspección cualitativa de texturas (cmap jet)
Muestras del régimen no saturado: Referencia vs Generada vs |Diferencia|.


In [ ]:
show_samples(imgs_gen, imgs_ref_norm, y_eval,
             label=f"Fidelidad de textura · régimen NO saturado (Hex<={HEX_SAT})",
             n_tex=6, n_sat=2, fname="step3_qualitative_texture_jet.png")


---
## Plots finales — Original externo vs Generado (cmap jet)
Mismos paneles que `ciclo_external_fixes` (filas alternas Original/Generado), color del
título por régimen respecto a **HEX_SAT = 0.80** (azul = texturado, rojo = saturado).
1. **Hex descendente** (saturado arriba → texturado abajo: muestra la transición física)
2. **Muestras aleatorias**


In [ ]:
# ── 50 imágenes ordenadas por Hex DESCENDENTE: Original vs Generado ───────────
rng_plot = np.random.RandomState(99)
n_show = 50
plot_idx = rng_plot.choice(N_EVAL, n_show, replace=False)
plot_idx = plot_idx[np.argsort(y_eval[plot_idx, HEX_IDX])[::-1]]   # Hex descendente

ncols = 10; nrows = n_show // ncols
kw = dict(cmap="jet", vmin=-1, vmax=1, interpolation="nearest")
fig, axes = plt.subplots(nrows*2, ncols, figsize=(ncols*1.9, nrows*2*1.9),
                         gridspec_kw={"hspace": 0.05, "wspace": 0.04})
fig.suptitle(
    "50 muestras — Original externo (fila impar) vs Generado (fila par)\n"
    f"Hex DESCENDENTE  ·  cmap jet  ·  azul=texturado (Hex<={HEX_SAT})  rojo=saturado",
    fontsize=11, fontweight="bold", y=1.01)
for col, idx in enumerate(plot_idx):
    ro = (col // ncols) * 2; rg = ro + 1; c = col % ncols
    ref = imgs_ref_norm[idx]; gen = imgs_gen[idx, :39, :39]
    hx = y_eval[idx, HEX_IDX]; rc = "tomato" if hx > HEX_SAT else "steelblue"
    axes[ro, c].imshow(ref, **kw); axes[rg, c].imshow(gen, **kw)
    axes[ro, c].set_title(f"Hex={hx:.2f}", fontsize=6, color=rc, pad=2)
    axes[rg, c].set_xlabel(f"MAE={float(np.abs(ref-gen).mean()):.2f}\nSSIM={ssim_fn(ref,gen,data_range=2.0):.2f}",
                           fontsize=5.5, labelpad=2)
    for ax in [axes[ro, c], axes[rg, c]]: ax.set_xticks([]); ax.set_yticks([])
for pair in range(nrows):
    axes[pair*2,   0].set_ylabel("Original",  fontsize=8, rotation=90, labelpad=3)
    axes[pair*2+1, 0].set_ylabel("Generado", fontsize=8, rotation=90, labelpad=3)
cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7])
fig.colorbar(plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(vmin=-1, vmax=1)), cax=cbar_ax, label="⟨sz⟩")
plt.savefig("final_50_hex_descent.png", dpi=150, bbox_inches="tight")
print("Guardado: final_50_hex_descent.png"); plt.show()


In [ ]:
# ── 50 imágenes aleatorias (sin orden): Original vs Generado ──────────────────
rng_plot2 = np.random.RandomState(42)
plot_idx2 = rng_plot2.choice(N_EVAL, n_show, replace=False)
fig, axes = plt.subplots(nrows*2, ncols, figsize=(ncols*1.9, nrows*2*1.9),
                         gridspec_kw={"hspace": 0.05, "wspace": 0.04})
fig.suptitle(
    "50 muestras aleatorias — Original externo vs Generado\n"
    f"cmap jet  ·  azul=texturado (Hex<={HEX_SAT})  rojo=saturado",
    fontsize=11, fontweight="bold", y=1.01)
for col, idx in enumerate(plot_idx2):
    ro = (col // ncols) * 2; rg = ro + 1; c = col % ncols
    ref = imgs_ref_norm[idx]; gen = imgs_gen[idx, :39, :39]
    hx = y_eval[idx, HEX_IDX]; rc = "tomato" if hx > HEX_SAT else "steelblue"
    axes[ro, c].imshow(ref, **kw); axes[rg, c].imshow(gen, **kw)
    axes[ro, c].set_title(f"Hex={hx:.2f}", fontsize=6, color=rc, pad=2)
    axes[rg, c].set_xlabel(f"MAE={float(np.abs(ref-gen).mean()):.2f}\nSSIM={ssim_fn(ref,gen,data_range=2.0):.2f}",
                           fontsize=5.5, labelpad=2)
    for ax in [axes[ro, c], axes[rg, c]]: ax.set_xticks([]); ax.set_yticks([])
for pair in range(nrows):
    axes[pair*2,   0].set_ylabel("Original",  fontsize=8, rotation=90, labelpad=3)
    axes[pair*2+1, 0].set_ylabel("Generado", fontsize=8, rotation=90, labelpad=3)
cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7])
fig.colorbar(plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(vmin=-1, vmax=1)), cax=cbar_ax, label="⟨sz⟩")
plt.savefig("final_50_random.png", dpi=150, bbox_inches="tight")
print("Guardado: final_50_random.png"); plt.show()
